# Appendix A3 (draft) — R1 손실 집중성 정량 근거

> **목적**: 논문 §5.2 (3)에서 인용한 R1 구간 LSTM-vs-ANN 비교 수치들의 **재현 가능한 정량 근거** 를 제공.
>
> 본 노트북에서 출력되는 모든 수치는 결론 5.2 (3) 본문에 그대로 인용 가능.

## 본 노트북이 재현하는 5.2 (3) 인용 수치

| 인용 수치 | 본 노트북 출처 |
|---|---|
| `p_w=mcap` LSTM 우위 0/15, ANN 우위 15/15 | §4 win-count 표 |
| `p_w=eq` LSTM 우위 1/15, ANN 우위 14/15 | §4 win-count 표 |
| `p_w=rp` LSTM 우위 14/15 | §4 win-count 표 |
| `p_w=mcap` R1 누적 손실 −7.75% | §5 cumulative ΔReturn 표 |
| 2011-03·04·08 시점 집중 76% | §7 concentration 계산 |
| 2011-08 단일 월 견인 40% | §7 concentration 계산 |

## 본 노트북은 논문에 직접 포함되지 않음

Appendix A3 본문은 논문에서 제거하고, 핵심 수치만 결론 5.2 (3)에 자기완결형으로 통합되었다 (논문 작성 결정). 본 노트북은 *팀 내부 검증·재현용 evidence backup* 으로만 보관.

## §0. 환경 셋업

In [34]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북이 appendix/에서 실행되든 final_pt/에서 실행되든 동작
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR if (NB_DIR / 'results').exists() else NB_DIR.parent
RESULTS_DIR = PROJECT_DIR / 'results'
DATA_DIR    = PROJECT_DIR / 'data'

assert RESULTS_DIR.exists(), f'results/ not found in {PROJECT_DIR}'
assert DATA_DIR.exists(),    f'data/ not found in {PROJECT_DIR}'

# R1 회복 구간 (논문 기준)
R1_START = '2010-01-01'
R1_END   = '2012-06-30'

# 시점 집중 분석 대상 월 (논문 5.2 (3) 인용)
CONCENTRATION_MONTHS = ['2011-03', '2011-04', '2011-08']
MAIN_EVENT_MONTH     = '2011-08'

# 99_main_analysis Cell 10·16과 동일 슬롯 집합 만들기 위한 필터
# (5 q-mode × 3 prior × 1 omega = 15 slot per p_w)
EXCLUDE_Q    = {'fix'}  # 99_main_analysis가 q_fix 제외하고 5 q-mode만 사용
OMEGA_FILTER = 'pap'    # 99_main_analysis Cell 16의 ΔReturn 표가 omega=pap 기준

print(f'PROJECT_DIR  : {PROJECT_DIR}')
print(f'R1 기간      : {R1_START} ~ {R1_END}')
print(f'집중 분석 월 : {CONCENTRATION_MONTHS}')
print(f'슬롯 필터    : q ∉ {EXCLUDE_Q}, omega == {OMEGA_FILTER!r}')

PROJECT_DIR  : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/low_vol_with_lstm_har_bl/final_pt
R1 기간      : 2010-01-01 ~ 2012-06-30
집중 분석 월 : ['2011-03', '2011-04', '2011-08']
슬롯 필터    : q ∉ {'fix'}, omega == 'pap'


## §1. 슬롯 파싱 및 pkl 로딩 (LSTM ↔ ANN 페어 매칭)

`mat_{prior}_{p_w}_{q}_{omega}[_ann].pkl` 패턴 파싱 후, LSTM/ANN 슬롯을 (prior, p_w, q, omega) 기준 페어 매칭.

**필터 적용**: §0에서 정의한 `EXCLUDE_Q={'fix'}`와 `OMEGA_FILTER='pap'`을 적용하여 99_main_analysis Cell 10·16과 동일한 5 q-mode × 3 prior × 1 omega = **15 slot per p_w** 집합을 구성. 이 필터는 결론 5.2 (3) 인용 수치(15/15, 14/15 등)와 정합하기 위함.

In [35]:
PATTERN = re.compile(
    r'^mat_(?P<prior>[a-z]+)_(?P<p_w>[a-z]+)_(?P<q>[a-z]+)_(?P<omega>[a-z]+)(?P<ann>_ann)?\.pkl$'
)

def parse_slot(filename):
    m = PATTERN.match(filename)
    if not m:
        return None
    return {
        'prior': m.group('prior'),
        'p_w':   m.group('p_w'),
        'q':     m.group('q'),
        'omega': m.group('omega'),
        'model': 'ANN' if m.group('ann') else 'LSTM',
    }

slot_records = []
for pkl_path in sorted(RESULTS_DIR.glob('*.pkl')):
    info = parse_slot(pkl_path.name)
    if info is None:
        continue
    with open(pkl_path, 'rb') as f:
        info['data'] = pickle.load(f)
    info['path'] = pkl_path
    slot_records.append(info)

def slot_key(s):
    return (s['prior'], s['p_w'], s['q'], s['omega'])

lstm_map = {slot_key(s): s for s in slot_records if s['model'] == 'LSTM'}
ann_map  = {slot_key(s): s for s in slot_records if s['model'] == 'ANN'}

# 페어 매칭 후 q_fix 제외 + omega='pap' 필터 적용
paired_keys = sorted(
    k for k in (set(lstm_map.keys()) & set(ann_map.keys()))
    if k[2] not in EXCLUDE_Q and k[3] == OMEGA_FILTER
)

print(f'로드 슬롯       : LSTM {len(lstm_map)} / ANN {len(ann_map)}')
print(f'페어 매칭 슬롯 (필터 후) : {len(paired_keys)}  (기대값: 3 prior × 5 q × 3 p_w × 1 omega = 45)')

# p_w별 슬롯 분포 (각 p_w가 정확히 15개여야 함 — prior 3 × q 5 × omega 1 = 15)
df_pairs = pd.DataFrame([{'prior': k[0], 'p_w': k[1], 'q': k[2], 'omega': k[3]} for k in paired_keys])
print(f'\np_w별 페어 수 (각 15 기대):\n{df_pairs["p_w"].value_counts().to_string()}')
print(f'\nq-mode 분포 (각 9 기대, fix 제외):\n{df_pairs["q"].value_counts().to_string()}')

로드 슬롯       : LSTM 108 / ANN 108
페어 매칭 슬롯 (필터 후) : 45  (기대값: 3 prior × 5 q × 3 p_w × 1 omega = 45)

p_w별 페어 수 (각 15 기대):
p_w
eq      15
mcap    15
rp      15

q-mode 분포 (각 9 기대, fix 제외):
q
fpm    9
inv    9
lam    9
raw    9
vsp    9


## §2. 무위험 수익률 (rf_monthly) 로딩

Sharpe 계산에 사용. `data/monthly_panel.csv`의 `rf_1m` 컬럼.

In [36]:
panel = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
rf_monthly = panel.groupby('date')['rf_1m'].first()
print(f'rf_monthly : {len(rf_monthly)}개월, {rf_monthly.index[0].date()} ~ {rf_monthly.index[-1].date()}')
print(f'R1 기간 rf : 평균 {rf_monthly.loc[R1_START:R1_END].mean():.5f} (월), 합계 {rf_monthly.loc[R1_START:R1_END].sum():.4f}')

rf_monthly : 252개월, 2005-01-31 ~ 2025-12-31
R1 기간 rf : 평균 0.00007 (월), 합계 0.0021


## §3. R1 구간 Sharpe ratio 계산 (슬롯별, LSTM vs ANN)

표준 정의: `SR = mean(excess) / std(returns) × √12`, 여기서 `excess = ret - rf`.

본 노트북에서 계산한 Sharpe가 `99_main_analysis.ipynb` Cell 10의 win-count(mcap 0/15, eq 1/15, rp 14/15)와 일치하는지로 정합성 검증.

In [37]:
def sharpe_r1(ret_series: pd.Series) -> float:
    """R1 구간 Sharpe ratio. ret_series는 월별 net return."""
    r1 = ret_series[(ret_series.index >= R1_START) & (ret_series.index <= R1_END)]
    if len(r1) < 2:
        return np.nan
    excess = r1 - rf_monthly.reindex(r1.index)
    if excess.std() == 0 or pd.isna(excess.std()):
        return np.nan
    return float(excess.mean() / r1.std() * np.sqrt(12))

r1_records = []
for key in paired_keys:
    lstm_data = lstm_map[key]['data']
    ann_data  = ann_map[key]['data']
    sr_l = sharpe_r1(lstm_data['ret'])
    sr_a = sharpe_r1(ann_data['ret'])
    r1_records.append({
        'prior': key[0], 'p_w': key[1], 'q': key[2], 'omega': key[3],
        'sr_lstm': sr_l, 'sr_ann': sr_a, 'sr_diff': sr_l - sr_a,
        'lstm_wins': sr_l > sr_a,
    })

df_r1 = pd.DataFrame(r1_records)
print(f'R1 슬롯별 Sharpe 계산 완료: {len(df_r1)}개 슬롯 (기대값: 45 = 3 prior × 5 q × 3 p_w)')
df_r1.head()

R1 슬롯별 Sharpe 계산 완료: 45개 슬롯 (기대값: 45 = 3 prior × 5 q × 3 p_w)


,prior,p_w,q,omega,sr_lstm,sr_ann,sr_diff,lstm_wins
0,eq,eq,fpm,pap,1.019744,1.122716,-0.102972,False
1,eq,eq,inv,pap,1.329169,1.336515,-0.007346,False
2,eq,eq,lam,pap,1.162697,1.202755,-0.040059,False
3,eq,eq,raw,pap,1.164155,1.210126,-0.045971,False
4,eq,eq,vsp,pap,1.220992,1.246973,-0.025981,False


## §4. ★ R1 win-count by p_w (5.2 (3) 인용 수치 #1)

p_w별로 LSTM이 ANN을 Sharpe 기준 능가한 슬롯 수.

In [38]:
win_table = (
    df_r1.groupby('p_w')['lstm_wins']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'LSTM_wins', 'count': 'total'})
)
win_table['ANN_wins']     = win_table['total'] - win_table['LSTM_wins']
win_table['LSTM_win_pct'] = win_table['LSTM_wins'] / win_table['total']
win_table = win_table[['LSTM_wins', 'ANN_wins', 'total', 'LSTM_win_pct']]

print('=' * 70)
print('R1 회복 구간 — LSTM Sharpe 우위 슬롯 수 (p_w별)')
print('=' * 70)
print(win_table.to_string())
print()
print('5.2 (3) 인용 정합 체크:')
for pw, expected_lstm_wins, denom in [('mcap', 0, 15), ('eq', 1, 15), ('rp', 14, 15)]:
    actual_l = int(win_table.loc[pw, 'LSTM_wins']) if pw in win_table.index else None
    actual_t = int(win_table.loc[pw, 'total'])     if pw in win_table.index else None
    status = '✓ 일치' if actual_l == expected_lstm_wins else f'⚠ 불일치 (논문 {expected_lstm_wins}/{denom})'
    print(f'  p_w={pw}: {actual_l}/{actual_t}  {status}')

R1 회복 구간 — LSTM Sharpe 우위 슬롯 수 (p_w별)
      LSTM_wins  ANN_wins  total  LSTM_win_pct
p_w                                           
eq            1        14     15      0.066667
mcap          0        15     15      0.000000
rp           14         1     15      0.933333

5.2 (3) 인용 정합 체크:
  p_w=mcap: 0/15  ✓ 일치
  p_w=eq: 1/15  ✓ 일치
  p_w=rp: 14/15  ✓ 일치


## §5. ★ R1 누적 ΔReturn by p_w (5.2 (3) 인용 수치 #2)

각 슬롯의 월별 LSTM−ANN 수익률 차이를 p_w 그룹 평균(15 슬롯 평균) 후, R1 구간 누적.

In [39]:
# 슬롯별 월별 ΔReturn 시계열
# 중요: 99_main_analysis Cell 16과 정합하려면 d.index를 한 달 앞으로 shift 해야 함
# (BL의 결정 시점 t의 ret는 실현 시점 t+1로 정렬하는 convention)
monthly_diff_records = []
for key in paired_keys:
    lstm_ret = lstm_map[key]['data']['ret'].dropna()
    ann_ret  = ann_map[key]['data']['ret'].dropna()
    common_idx = lstm_ret.index.intersection(ann_ret.index)
    if len(common_idx) == 0:
        continue
    diff = lstm_ret.loc[common_idx] - ann_ret.loc[common_idx]
    # ★ Calendar shift: 결정 시점 → 실현 시점 (한 달 forward)
    diff.index = diff.index + pd.offsets.MonthEnd(1)
    # R1 구간 mask 적용 (shift 후)
    r1_mask = (diff.index >= R1_START) & (diff.index <= R1_END)
    diff = diff[r1_mask]
    for dt, v in diff.items():
        monthly_diff_records.append({
            'prior': key[0], 'p_w': key[1], 'q': key[2], 'omega': key[3],
            'date': dt, 'diff': float(v),
        })
df_monthly = pd.DataFrame(monthly_diff_records)

# p_w 그룹 — 같은 p_w 내 모든 슬롯의 월별 ΔReturn 평균 → 월별 1개 시계열
pw_monthly = df_monthly.groupby(['p_w', 'date'])['diff'].mean().unstack('p_w')

# R1 누적 (해당 기간 월별 ΔReturn 합)
cum_r1 = pw_monthly.sum().rename('cumulative_diff').to_frame()
cum_r1['cumulative_pct'] = cum_r1['cumulative_diff'] * 100

print('=' * 70)
print('R1 회복 구간 — p_w별 누적 LSTM−ANN ΔReturn (15 슬롯 평균, calendar shift 적용)')
print('=' * 70)
print(cum_r1.round(4).to_string())
print()
print('5.2 (3) 인용 정합 체크:')
for pw, expected in [('mcap', -0.0775), ('eq', -0.0174), ('rp', +0.0019)]:
    actual = cum_r1.loc[pw, 'cumulative_diff'] if pw in cum_r1.index else None
    if actual is not None:
        match = abs(actual - expected) < 0.001
        status = '✓ 일치' if match else f'⚠ 불일치 (논문 {expected:+.4f})'
        print(f'  p_w={pw}: {actual:+.4f}  {status}')

R1 회복 구간 — p_w별 누적 LSTM−ANN ΔReturn (15 슬롯 평균, calendar shift 적용)
      cumulative_diff  cumulative_pct
p_w                                  
eq            -0.0174         -1.7391
mcap          -0.0775         -7.7468
rp             0.0019          0.1889

5.2 (3) 인용 정합 체크:
  p_w=mcap: -0.0775  ✓ 일치
  p_w=eq: -0.0174  ✓ 일치
  p_w=rp: +0.0019  ✓ 일치


## §6. R1 월별 ΔReturn 정렬 (worst → best)

`p_w=mcap`을 중심으로 R1 30개월 ΔReturn 정렬, 최악 5개 월 식별.

In [40]:
pw_focus = 'mcap'
mcap_monthly = pw_monthly[pw_focus].dropna().sort_values()

print('=' * 70)
print(f'R1 회복 구간 — p_w={pw_focus} 월별 ΔReturn 정렬 (worst 10)')
print('=' * 70)
worst10 = mcap_monthly.head(10).to_frame('diff')
worst10['diff_pct'] = worst10['diff'] * 100
print(worst10.round(4).to_string())
print()
print(f'전체 R1 월 수: {len(mcap_monthly)}')
print(f'음수 월 수   : {(mcap_monthly < 0).sum()}')
print(f'양수 월 수   : {(mcap_monthly > 0).sum()}')

R1 회복 구간 — p_w=mcap 월별 ΔReturn 정렬 (worst 10)
              diff  diff_pct
date                        
2011-08-31 -0.0308   -3.0778
2011-04-30 -0.0165   -1.6458
2011-03-31 -0.0119   -1.1865
2010-04-30 -0.0082   -0.8205
2010-08-31 -0.0051   -0.5142
2012-05-31 -0.0028   -0.2766
2010-07-31 -0.0027   -0.2709
2010-02-28 -0.0026   -0.2559
2011-12-31 -0.0021   -0.2081
2011-05-31 -0.0019   -0.1852

전체 R1 월 수: 29
음수 월 수   : 19
양수 월 수   : 10


## §7. ★ Concentration 계산 (5.2 (3) 인용 수치 #3, #4)

- **76%**: 2011-03·04·08 세 시점 손실이 R1 mcap 누적 손실에서 차지하는 비율
- **40%**: 2011-08 단일 월이 R1 mcap 누적 손실에서 차지하는 비율

In [41]:
# 대상 월 timestamp 변환
def to_month_end(ym: str) -> pd.Timestamp:
    return pd.to_datetime(ym) + pd.offsets.MonthEnd(0)

concentration_indices = [to_month_end(ym) for ym in CONCENTRATION_MONTHS]
main_event_index      = to_month_end(MAIN_EVENT_MONTH)

# 누적 손실
cumulative_mcap = mcap_monthly.sum()

# 3개 월 합
top3_values = mcap_monthly.reindex(concentration_indices)
top3_sum    = top3_values.sum()

# 2011-08 단일
main_event_value = mcap_monthly.get(main_event_index, np.nan)

# 비율
concentration_top3 = top3_sum / cumulative_mcap
concentration_main = main_event_value / cumulative_mcap

print('=' * 70)
print('R1 mcap 누적 손실 — 시점 집중성 분석')
print('=' * 70)
print(f'R1 mcap 누적 LSTM−ANN ΔReturn : {cumulative_mcap:+.4f}  ({cumulative_mcap*100:+.2f}%)')
print()
print('2011-03·04·08 세 시점:')
for dt, v in top3_values.items():
    print(f'  {dt.strftime("%Y-%m")} : {v:+.4f}  ({v*100:+.2f}%)')
print(f'  ---------------------------------')
print(f'  합계         : {top3_sum:+.4f}  ({top3_sum*100:+.2f}%)')
print()
print(f'★ 시점 집중성 (top3 / cumulative)        : {concentration_top3*100:.1f}%')
print(f'★ 2011-08 단일 견인 ({MAIN_EVENT_MONTH} / cumulative) : {concentration_main*100:.1f}%')
print()
print('5.2 (3) 인용 정합 체크:')
print(f'  논문 인용 "76%"   ↔ 본 노트북: {concentration_top3*100:.1f}%  '
      f'{"✓ 일치" if abs(concentration_top3*100 - 76.0) < 2 else "⚠ 불일치"}')
print(f'  논문 인용 "40%"   ↔ 본 노트북: {concentration_main*100:.1f}%  '
      f'{"✓ 일치" if abs(concentration_main*100 - 40.0) < 2 else "⚠ 불일치"}')

R1 mcap 누적 손실 — 시점 집중성 분석
R1 mcap 누적 LSTM−ANN ΔReturn : -0.0775  (-7.75%)

2011-03·04·08 세 시점:
  2011-03 : -0.0119  (-1.19%)
  2011-04 : -0.0165  (-1.65%)
  2011-08 : -0.0308  (-3.08%)
  ---------------------------------
  합계         : -0.0591  (-5.91%)

★ 시점 집중성 (top3 / cumulative)        : 76.3%
★ 2011-08 단일 견인 (2011-08 / cumulative) : 39.7%

5.2 (3) 인용 정합 체크:
  논문 인용 "76%"   ↔ 본 노트북: 76.3%  ✓ 일치
  논문 인용 "40%"   ↔ 본 노트북: 39.7%  ✓ 일치


## §8. 종합 — 5.2 (3) 인용 수치 한 표 정리

본 노트북이 출력한 모든 수치를 한 표로 정리. 논문 본문 인용 시 이 표가 정합성 근거.

In [42]:
summary_rows = [
    {'category': 'win-count',     'metric': 'p_w=mcap LSTM 우위',     '본 노트북': f"{int(win_table.loc['mcap', 'LSTM_wins'])}/{int(win_table.loc['mcap', 'total'])}",   '논문 5.2 (3) 인용': '0/15 (mcap)'},
    {'category': 'win-count',     'metric': 'p_w=eq LSTM 우위',       '본 노트북': f"{int(win_table.loc['eq',   'LSTM_wins'])}/{int(win_table.loc['eq',   'total'])}",   '논문 5.2 (3) 인용': '1/15 (eq)'},
    {'category': 'win-count',     'metric': 'p_w=rp LSTM 우위',       '본 노트북': f"{int(win_table.loc['rp',   'LSTM_wins'])}/{int(win_table.loc['rp',   'total'])}",   '논문 5.2 (3) 인용': '14/15 (rp)'},
    {'category': 'cumulative',    'metric': 'p_w=mcap 누적 ΔReturn',  '본 노트북': f"{cum_r1.loc['mcap', 'cumulative_diff']*100:+.2f}%", '논문 5.2 (3) 인용': '−7.75%'},
    {'category': 'cumulative',    'metric': 'p_w=eq 누적 ΔReturn',    '본 노트북': f"{cum_r1.loc['eq',   'cumulative_diff']*100:+.2f}%", '논문 5.2 (3) 인용': '−1.74%'},
    {'category': 'cumulative',    'metric': 'p_w=rp 누적 ΔReturn',    '본 노트북': f"{cum_r1.loc['rp',   'cumulative_diff']*100:+.2f}%", '논문 5.2 (3) 인용': '+0.19%'},
    {'category': 'concentration', 'metric': '2011-03·04·08 집중',     '본 노트북': f"{concentration_top3*100:.1f}%",                       '논문 5.2 (3) 인용': '≈ 76%'},
    {'category': 'concentration', 'metric': '2011-08 단일 견인',      '본 노트북': f"{concentration_main*100:.1f}%",                       '논문 5.2 (3) 인용': '≈ 40%'},
]
df_summary = pd.DataFrame(summary_rows)
print('=' * 90)
print('★ Appendix A3 evidence — 결론 5.2 (3) 인용 수치 정합성 표')
print('=' * 90)
print(df_summary.to_string(index=False))

★ Appendix A3 evidence — 결론 5.2 (3) 인용 수치 정합성 표
     category              metric  본 노트북 논문 5.2 (3) 인용
    win-count    p_w=mcap LSTM 우위   0/15   0/15 (mcap)
    win-count      p_w=eq LSTM 우위   1/15     1/15 (eq)
    win-count      p_w=rp LSTM 우위  14/15    14/15 (rp)
   cumulative p_w=mcap 누적 ΔReturn -7.75%        −7.75%
   cumulative   p_w=eq 누적 ΔReturn -1.74%        −1.74%
   cumulative   p_w=rp 누적 ΔReturn +0.19%        +0.19%
concentration    2011-03·04·08 집중  76.3%         ≈ 76%
concentration       2011-08 단일 견인  39.7%         ≈ 40%


## §9. 참고 — 잠재적 정합성 이슈

- **Sharpe 정의**: 본 노트북은 표준 정의 `SR = mean(excess) / std(ret) × √12` 사용. `99_main_analysis.ipynb`의 Sharpe 정의가 다를 경우 §4 win-count가 1~2개 슬롯 차이 가능. §4 끝부분의 "정합 체크" 출력을 확인하고, 불일치 시 분자(excess vs raw)·분모(ret std vs excess std)·연환산 factor를 조정.
- **누적 ΔReturn**: `99_main_analysis.ipynb` Cell 7443~7465의 월별 ΔReturn 표와 일치해야 함. §5 끝부분의 정합 체크 사용.
- **Concentration**: 누적 ΔReturn이 일치하면 자동 일치. §7 결과 직접 비교.